In [190]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk #manejo de stopwords
from nltk.corpus import stopwords
import spacy
import textwrap

pd.set_option('display.max_columns', None)

In [191]:
df = pd.read_csv('spotify_songs.csv')

In [192]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18454 entries, 0 to 18453
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   track_id                  18454 non-null  object 
 1   track_name                18454 non-null  object 
 2   track_artist              18454 non-null  object 
 3   lyrics                    18194 non-null  object 
 4   track_popularity          18454 non-null  int64  
 5   track_album_id            18454 non-null  object 
 6   track_album_name          18454 non-null  object 
 7   track_album_release_date  18454 non-null  object 
 8   playlist_name             18454 non-null  object 
 9   playlist_id               18454 non-null  object 
 10  playlist_genre            18454 non-null  object 
 11  playlist_subgenre         18454 non-null  object 
 12  danceability              18454 non-null  float64
 13  energy                    18454 non-null  float64
 14  key   

In [193]:
df.head()

,track_id,track_name,track_artist,lyrics,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,playlist_genre,playlist_subgenre,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,language
0,0017A6SJgTbfQVU2EtsPNo,Pangarap,Barbie's Cradle,Minsan pa Nang ako'y napalingon Hindi ko alam ...,41,1srJQ0njEQgd8w4XSqI4JQ,Trip,2001-01-01,Pinoy Classic Rock,37i9dQZF1DWYDQ8wBxd7xt,rock,classic rock,0.682,0.401,2,-10.068,1,0.0236,0.27900,0.01170,0.0887,0.566,97.091,235440,tl
1,004s3t0ONYlzxII9PLgU6z,I Feel Alive,Steady Rollin,"The trees, are singing in the wind The sky blu...",28,3z04Lb9Dsilqw68SHt6jLB,Love & Loss,2017-11-21,Hard Rock Workout,3YouF0u7waJnolytf9JCXf,rock,hard rock,0.303,0.880,9,-4.739,1,0.0442,0.01170,0.00994,0.3470,0.404,135.225,373512,en
2,00chLpzhgVjxs1zKC9UScL,Poison,Bell Biv DeVoe,"NA Yeah, Spyderman and Freeze in full effect U...",0,6oZ6brjB8x3GoeSYdwJdPc,Gold,2005-01-01,"Back in the day - R&B, New Jack Swing, Swingbe...",3a9y4eeCJRmG9p4YKfqYIx,r&b,new jack swing,0.845,0.652,6,-7.504,0,0.2160,0.00432,0.00723,0.4890,0.650,111.904,262467,en
3,00cqd6ZsSkLZqGMlQCR0Zo,Baby It's Cold Outside (feat. Christina Aguilera),CeeLo Green,I really can't stay Baby it's cold outside I'v...,41,3ssspRe42CXkhPxdc12xcp,CeeLo's Magic Moment,2012-10-29,Christmas Soul,6FZYc2BvF7tColxO8PBShV,r&b,neo soul,0.425,0.378,5,-5.819,0,0.0341,0.68900,0.00000,0.0664,0.405,118.593,243067,en
4,00emjlCv9azBN0fzuuyLqy,Dumb Litty,KARD,Get up out of my business You don't keep me fr...,65,7h5X3xhh3peIK9Y0qI5hbK,KARD 2nd Digital Single ‘Dumb Litty’,2019-09-22,K-Party Dance Mix,37i9dQZF1DX4RDXswvP6Mj,pop,dance pop,0.760,0.887,9,-1.993,1,0.0409,0.03700,0.00000,0.1380,0.240,130.018,193160,en


##### Para comenzar nos concentraremos en las letras de las canciones, por lo que en un principio consideraremos las columnas correspondientes a: nombre del artista, nombre de la canción, género musical y letra

In [194]:
df = df[['track_name', 'track_artist', 'lyrics', 'playlist_genre', 'language']]

In [195]:
df.head()

,track_name,track_artist,lyrics,playlist_genre,language
0,Pangarap,Barbie's Cradle,Minsan pa Nang ako'y napalingon Hindi ko alam ...,rock,tl
1,I Feel Alive,Steady Rollin,"The trees, are singing in the wind The sky blu...",rock,en
2,Poison,Bell Biv DeVoe,"NA Yeah, Spyderman and Freeze in full effect U...",r&b,en
3,Baby It's Cold Outside (feat. Christina Aguilera),CeeLo Green,I really can't stay Baby it's cold outside I'v...,r&b,en
4,Dumb Litty,KARD,Get up out of my business You don't keep me fr...,pop,en


In [196]:
df.isnull().sum()

track_name          0
track_artist        0
lyrics            260
playlist_genre      0
language          260
dtype: int64

##### Como nos interesa aplicar svd sobre la letra de las canciones, eliminaremos las entradas que no tienen letra.

In [197]:
df = df.dropna(subset=['lyrics'])

In [198]:
df.isnull().sum()

track_name        0
track_artist      0
lyrics            0
playlist_genre    0
language          0
dtype: int64

In [199]:
df['playlist_genre'].unique()

array(['rock', 'r&b', 'pop', 'edm', 'latin', 'rap'], dtype=object)

##### Para este proyecto, nos gustaría hacer un análisis semántico, particularmente de la letra de las canciones cuyo género musical es el rap y están en español. Por lo que reduciremos el dataset a este género.

In [200]:
df = df[(df['playlist_genre'] == 'rap') & (df['language'] == 'es')]

In [201]:
df.head()

,track_name,track_artist,lyrics,playlist_genre,language
53,Generé,Bipo Montana,Desde abajo hacia arriba me elevé (Me elevé) A...,rap,es
109,She Don't Give a Fo,Duki,NA Ey (Duki) Yeah (Khea Young Flex) Ah-ooou-oh...,rap,es
143,Fvck Luv,Duki,"NA (Yeah, yeah, yeah) Bae-ae Yeah, yeah, color...",rap,es
178,Déjame Volar,Aleman,NA La segunda clave para el éxito en este nego...,rap,es
217,No Te Quiero Perder,Sabino,NA No te quiero perder Que la distancia se dev...,rap,es


In [202]:
df['lyrics'] = df['lyrics'].str.strip('NA')


## P2. Hipótesis Inicial:
##### Como grupo pensamos que en general, un tema del que se escribe mucho en la música popular independiente de su género es el amor y el desamor, por lo que esperamos que palabras relacionadas a esos temas aparezcan como relevantes. Además, en latinoamérica es común en las letras de canciones urbanas, la temática de 'alardeo', es decir, incorporar frases relacionadas con el éxito, el dinero, llegar a la cima, ganar, perder, entre otras. Por lo que tambien esperamos que estas palabras aparezcan. 

## P3. Preprocesamiento del texto
### Explicación: 
##### Para el preprocesamiento de los datos, utilizamos como guía el código del notebook de ejemplo que se mostró en clases con los archivos del curso ('lsa_corpus_curso.html').

##### La lógica del procedimiento a seguir es la siguiente:
- Se utilizó el modelo 'es_core_news_sm' de la librería spacy que permite separar el texto en tokens (o pequeños fragmentos). Durante el proceso de limpieza se hizo una normalización a minúsculas (es decir, aplicar str.lower a cada fragmento) y también se filtraron todos los caracteres que no son letras (considere números, caracteres especiales, emojis, y similares). 

- Luego siguiendo el ejemplo visto en clases, se aplicó lematización para simplificar y unificar palabras similares bajo un mismo significado (por ejemplo, gané, ganando se combinan bajo una sola palabra 'ganar' y así para todas las palabras).

- También se eliminaron las stopwords, palabras como conectores, conjunciones y similares que no aportan mucho al análisis, así como los signos de puntuación. Para esto se utilizó la lista de stopwords que proporciona la librería spacy.

- Finalmente, se sigue la recomendación presentada en el enunciado de construir la matriz TF-IDF sobre el conjunto de datos ya limpio.

##### Justificación: El procesamiento elegido es razonable porque las letras de canciones, en especial aquellas que corresponden a improvisaciones o raps tienden a tener un alto número de conectores o palabras que podemos considerar como stopwords. Asimismo, dentro de la complejidad de los juegos de palabras en las letras de raps o improvisaciones, a menudo se prioriza la rima y la creatividad por encima de la coherencia, por lo que lematizar nos ayuda a 'estandarizar' o simplificar el amplio rango de vocabulario que se usa en este género musical. Con esto, procuramos eliminar el mayor ruido posible de los datos, ayudandonos para el análisis posterior.

In [203]:
!python -m spacy download es_core_news_sm

     ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
     -- ------------------------------------- 0.8/12.9 MB 9.0 MB/s eta 0:00:02
     --- ------------------------------------ 1.0/12.9 MB 5.5 MB/s eta 0:00:03
     ---- ----------------------------------- 1.6/12.9 MB 3.2 MB/s eta 0:00:04
     -------- ------------------------------- 2.6/12.9 MB 3.5 MB/s eta 0:00:03
     -------- ------------------------------- 2.6/12.9 MB 3.5 MB/s eta 0:00:03
     --------- ------------------------------ 3.1/12.9 MB 3.0 MB/s eta 0:00:04
     ----------- ---------------------------- 3.7/12.9 MB 2.9 MB/s eta 0:00:04
     ----------- ---------------------------- 3.7/12.9 MB 2.9 MB/s eta 0:00:04
     --------------- ------------------------ 5.0/12.9 MB 2.8 MB/s eta 0:00:03
     ----------------- ---------------------- 5.5/12.9 MB 2.9 MB/s eta 0:00:03
     -------------------- ------------------- 6.6/12.9 MB 2.9 MB/s eta 0:00:03
     ---------------------- ----------------- 7.3/12.9 MB 3

In [ ]:
nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])

# Función generada con Gemini con el propósito  de limpiar el texto de las letras de las canciones,
# esto incluye eliminar stopwords, signos de puntuación y tomar los lemas de las palabras.
def limpiar_texto(columna):
    textos_limpios = []
    
    # nlp.pipe es la forma más rápida de procesar muchas filas
    for doc in nlp.pipe(columna.astype(str), batch_size=50):
        # Filtramos: 
        # - token.lemma_ : toma la base de la palabra
        # - not token.is_stop : elimina stopwords
        # - not token.is_punct : elimina signos de puntuación
        # - token.is_alpha : asegura que sean letras (elimina números o símbolos raros)
        tokens = [token.lemma_.lower() for token in doc 
                  if not token.is_stop and not token.is_punct and token.is_alpha]
        
        # Volvemos a unir los lemas en una sola cadena de texto
        textos_limpios.append(" ".join(tokens))
        
    return textos_limpios

#Aplicamos esta función a la columna con las letras de las canciones
#seleccionadas y obtenemos una nueva columna con el texto limpio y preparado para el análisis.
df['cl_lyrics'] = limpiar_texto(df['lyrics']) 

In [ ]:
# Un ejemplo del resultado de la limpieza
print(df[['lyrics', 'cl_lyrics']].head(1))

                                               lyrics  \
53  Desde abajo hacia arriba me elevé (Me elevé) A...   

                                            cl_lyrics  
53  abajo elever elever loco dejar vender vender c...  


##### Ahora se construye la matriz TF-IDF sobre la columna cl_lyrics

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

In [ ]:
# Obtenemos las stopwords de spacy como un set y las convertimos a una lista 
# para usarlas en el vectorizador
stopwords = list(nlp.Defaults.stop_words)

In [ ]:
# Código rescatado del notebook de ejmplo con los apuntes del curso
vectorizer_final = TfidfVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b[^\W\d_]{3,}\b",
    stop_words=stopwords,
    min_df=2,
    max_df=0.8,
    ngram_range=(1, 2),
)

X_final = vectorizer_final.fit_transform(df['cl_lyrics'])
vocabulario_final = vectorizer_final.get_feature_names_out()
A_final = X_final.toarray().T

vectorizer_final_counts = CountVectorizer(vocabulary=vocabulario_final)
X_final_counts = vectorizer_final_counts.transform(df['cl_lyrics'])
frecuencias_final = np.asarray(X_final_counts.sum(axis=0)).ravel()

print("Forma de X_final (documentos x terminos):", X_final.shape)
print("Forma de A_final (terminos x documentos):", A_final.shape)
print("Primeros terminos del vocabulario final:", vocabulario_final[:30])
print("Unigramas:", sum(len(t.split()) == 1 for t in vocabulario_final))
print("Bigramas:", sum(len(t.split()) == 2 for t in vocabulario_final))


Forma de X_final (documentos x terminos): (343, 7142)
Forma de A_final (terminos x documentos): (7142, 343)
Primeros terminos del vocabulario final: ['aah' 'abajo' 'abajo colchón' 'abajo mañana' 'abajo tope' 'abandonado'
 'abandonar' 'abarca' 'abarca trir' 'abierto' 'abierto ventana' 'abismo'
 'abogada' 'abogado' 'abortar' 'abracadabra' 'abrazar' 'abrigo' 'abrir'
 'abrir ojo' 'abrir rompen' 'abuela' 'abundo' 'abundo rumbo' 'aburrido'
 'aburrir' 'abusa' 'abusa emplea' 'abuse' 'abuso']
Unigramas: 4185
Bigramas: 2957


## P4. Construcción de la Matriz textual